# Multi-Source Customer Support Agent

In this notebook, we build a customer support agent that can query both structured data (a SQLite database of customers, orders, and products) and unstructured data (a knowledge base of Markdown documents) to answer user questions using OpenAI's Responses API.

## Architecture Overview

The multi-source pattern gives our agent access to **two fundamentally different types of data**:

1. **Structured data (SQLite)** - Customer records, order history, and product inventory stored in relational tables. The agent queries this through parameterized functions (not raw SQL).
2. **Unstructured data (Knowledge Base)** - Markdown documents covering return policies, shipping info, FAQs, and pricing plans. The agent searches these by keyword.

The agent has **3 tools** at its disposal:

| Tool | Data Source | Purpose |
|------|------------|---------|
| `search_orders` | SQLite `orders` + `customers` | Look up orders by email, ID, or status |
| `search_products` | SQLite `products` | Find products by category, keyword, or stock |
| `search_knowledge_base` | Markdown docs | Search policies, FAQs, and documentation |

```
                    +-------------------+
                    |   User Question   |
                    +--------+----------+
                             |
                             v
                    +-------------------+
                    |   GPT-4o Agent    |
                    |  (Responses API)  |
                    +--------+----------+
                             |
              +--------------+--------------+
              |              |              |
              v              v              v
     +--------+---+ +------+------+ +------+--------+
     |search_orders| |search_prods | |search_kb      |
     +--------+---+ +------+------+ +------+--------+
              |              |              |
              v              v              v
     +--------+---+ +------+------+ +------+--------+
     |   SQLite    | |   SQLite    | | Markdown Docs |
     | orders/cust | |  products   | | (in-memory)   |
     +-------------+ +-------------+ +---------------+
```

The key insight is that the **model never generates SQL directly**. Instead, each tool accepts structured parameters (email, order ID, category, etc.) and the Python function constructs the appropriate query internally. This is safer and more predictable than text-to-SQL approaches.

In [ ]:
# DEPENDENCY: pip install -q openai
# (packages should be pre-installed in venv before running this script)

In [ ]:
import os
import json
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

client = OpenAI()
MODEL = "gpt-4o"

## Create the SQLite Database

We will create an in-memory SQLite database with three tables: `customers`, `orders`, and `products`. This simulates a real customer support backend without requiring any external files.

In [ ]:
import sqlite3

# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row  # Return rows as dictionaries
cursor = conn.cursor()

# ── Create tables ──────────────────────────────────────────────────────────

cursor.executescript("""
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    plan_type TEXT NOT NULL CHECK(plan_type IN ('free', 'pro', 'enterprise')),
    created_at TEXT NOT NULL
);

CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL,
    in_stock INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    product_name TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    total_price REAL NOT NULL,
    status TEXT NOT NULL CHECK(status IN ('pending', 'shipped', 'delivered', 'refunded')),
    created_at TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(id)
);
""")

# ── Insert sample customers ─────────────────────────────────────────────────

customers = [
    (1, "Alice Johnson",   "alice@example.com",   "pro",        "2024-01-15"),
    (2, "Bob Smith",       "bob@example.com",     "free",       "2024-02-20"),
    (3, "Carol Williams",  "carol@example.com",   "enterprise", "2023-11-05"),
    (4, "David Brown",     "david@example.com",   "pro",        "2024-03-10"),
    (5, "Eve Davis",       "eve@example.com",     "free",       "2024-04-01"),
    (6, "Frank Miller",    "frank@example.com",   "enterprise", "2023-09-22"),
    (7, "Grace Lee",       "grace@example.com",   "pro",        "2024-05-18"),
    (8, "Henry Wilson",    "henry@example.com",   "free",       "2024-06-30"),
]
cursor.executemany("INSERT INTO customers VALUES (?, ?, ?, ?, ?)", customers)

# ── Insert sample products ──────────────────────────────────────────────────

products = [
    (1,  "Wireless Mouse",       "Electronics",  29.99, 1),
    (2,  "Mechanical Keyboard",  "Electronics", 149.99, 1),
    (3,  "USB-C Hub",            "Electronics",  49.99, 1),
    (4,  "Laptop Stand",         "Accessories",  39.99, 1),
    (5,  "Webcam HD 1080p",      "Electronics",  79.99, 0),
    (6,  "Noise-Canceling Headphones", "Electronics", 199.99, 1),
    (7,  "Monitor Light Bar",    "Accessories",  54.99, 1),
    (8,  "Ergonomic Chair",      "Furniture",   399.99, 1),
    (9,  "Standing Desk",        "Furniture",   599.99, 0),
    (10, "Portable Charger",     "Electronics",  24.99, 1),
]
cursor.executemany("INSERT INTO products VALUES (?, ?, ?, ?, ?)", products)

# ── Insert sample orders ───────────────────────────────────────────────────

orders = [
    (1001, 1, "Wireless Mouse",       1,  29.99, "delivered",  "2024-07-01"),
    (1002, 1, "Mechanical Keyboard",  1, 149.99, "shipped",    "2024-08-15"),
    (1003, 2, "USB-C Hub",            2,  99.98, "pending",    "2024-09-01"),
    (1004, 3, "Ergonomic Chair",      1, 399.99, "delivered",  "2024-06-20"),
    (1005, 3, "Monitor Light Bar",    3, 164.97, "shipped",    "2024-09-10"),
    (1006, 4, "Noise-Canceling Headphones", 1, 199.99, "refunded", "2024-08-05"),
    (1007, 5, "Portable Charger",     2,  49.98, "delivered",  "2024-07-22"),
    (1008, 6, "Standing Desk",        1, 599.99, "pending",    "2024-09-15"),
    (1009, 7, "Laptop Stand",         1,  39.99, "shipped",    "2024-09-12"),
    (1010, 2, "Wireless Mouse",       1,  29.99, "delivered",  "2024-05-10"),
]
cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?)", orders)
conn.commit()

# Verify the data
for table in ["customers", "products", "orders"]:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")

## Create the Knowledge Base

Our knowledge base consists of Markdown documents stored as Python dictionaries. In a production system these would be files on disk or in a vector store, but for this demo we keep everything in memory.

In [ ]:
knowledge_base = {
    "return_policy.md": """# Return Policy

We offer a 30-day return policy on all products.

## Eligibility
- Items must be unused and in original packaging.
- Electronics must include all original accessories.
- Furniture items must be unassembled or in original condition.

## Process
1. Contact support with your order ID.
2. We will email you a prepaid return shipping label.
3. Ship the item within 7 days of receiving the label.
4. Refund is processed within 5 business days after we receive the item.

## Exceptions
- Sale items are final sale and cannot be returned.
- Customized products are non-returnable.
- Items damaged by the customer are not eligible for return.""",

    "shipping_info.md": """# Shipping Information

We ship to all 50 US states and select international destinations.

## Domestic Shipping
- **Standard shipping**: 5-7 business days, free on orders over $50.
- **Express shipping**: 2-3 business days, $9.99 flat rate.
- **Overnight shipping**: Next business day, $19.99 flat rate.

## International Shipping
- Available to Canada, UK, and EU countries.
- Delivery takes 10-15 business days.
- Customs fees are the responsibility of the buyer.

## Tracking
- All orders receive a tracking number via email once shipped.
- Track your order at our website or the carrier's website.""",

    "faq.md": """# Frequently Asked Questions

## How do I reset my password?
Go to the login page and click "Forgot Password". Enter your email address and we will send you a reset link. The link expires after 24 hours.

## How do I update my billing information?
Log in to your account, go to Settings > Billing, and update your payment method. Changes take effect on your next billing cycle.

## Can I change my plan?
Yes, you can upgrade or downgrade your plan at any time from Settings > Subscription. Upgrades take effect immediately; downgrades take effect at the end of your current billing period.

## How do I contact support?
You can reach us via this chat agent, by emailing support@example.com, or by calling 1-800-555-0199 during business hours (Mon-Fri, 9 AM - 6 PM EST).

## Do you offer bulk discounts?
Yes, Enterprise plan customers receive volume-based pricing. Contact our sales team at sales@example.com for a custom quote.""",

    "pricing_plans.md": """# Pricing Plans

## Free Plan
- 100 API calls per month
- Basic support (email only, 48-hour response time)
- Access to standard product catalog
- No bulk ordering

## Pro Plan - $29/month
- 10,000 API calls per month
- Priority support (email and chat, 4-hour response time)
- Access to full product catalog including early releases
- Bulk ordering with 10% discount
- Order history export (CSV)

## Enterprise Plan - Custom pricing
- Unlimited API calls
- Dedicated account manager
- 24/7 phone, email, and chat support
- Custom integrations and SLAs
- Volume-based pricing with up to 30% discount
- Advanced analytics dashboard"""
}

print(f"Knowledge base contains {len(knowledge_base)} documents:")
for filename, content in knowledge_base.items():
    line_count = len(content.strip().splitlines())
    print(f"  - {filename} ({line_count} lines)")

## Define Tool Functions

Each tool function accepts **structured parameters** and internally constructs the appropriate SQL query or search logic. The model never sees or generates raw SQL -- it simply passes parameters like `customer_email` or `category`, and the function handles the rest.

This is a deliberate design choice: parameterized queries are safer, more predictable, and easier to test than text-to-SQL approaches.

In [ ]:
def search_orders(customer_email=None, order_id=None, status=None):
    """
    Search for orders using structured parameters.
    Builds a SQL query internally based on the provided filters.
    """
    query = """
        SELECT o.id AS order_id, c.name AS customer_name, c.email,
               o.product_name, o.quantity, o.total_price,
               o.status, o.created_at
        FROM orders o
        JOIN customers c ON o.customer_id = c.id
        WHERE 1=1
    """
    params = []

    if customer_email:
        query += " AND c.email = ?"
        params.append(customer_email)
    if order_id is not None:
        query += " AND o.id = ?"
        params.append(order_id)
    if status:
        query += " AND o.status = ?"
        params.append(status)

    query += " ORDER BY o.created_at DESC"

    rows = cursor.execute(query, params).fetchall()
    results = [dict(row) for row in rows]

    if not results:
        return json.dumps({"message": "No orders found matching the criteria."})
    return json.dumps(results, indent=2)


def search_products(category=None, search_term=None, in_stock_only=False):
    """
    Search for products using structured parameters.
    Builds a SQL query internally based on the provided filters.
    """
    query = "SELECT id, name, category, price, in_stock FROM products WHERE 1=1"
    params = []

    if category:
        query += " AND LOWER(category) = LOWER(?)"
        params.append(category)
    if search_term:
        query += " AND LOWER(name) LIKE LOWER(?)"
        params.append(f"%{search_term}%")
    if in_stock_only:
        query += " AND in_stock = 1"

    query += " ORDER BY price ASC"

    rows = cursor.execute(query, params).fetchall()
    results = [
        {**dict(row), "in_stock": bool(dict(row)["in_stock"])}
        for row in rows
    ]

    if not results:
        return json.dumps({"message": "No products found matching the criteria."})
    return json.dumps(results, indent=2)


def search_knowledge_base(query):
    """
    Simple keyword search across all knowledge base documents.
    Splits the query into words and returns documents that contain
    any of those keywords (case-insensitive).
    """
    keywords = query.lower().split()
    matches = []

    for filename, content in knowledge_base.items():
        content_lower = content.lower()
        # Count how many query keywords appear in this document
        matched_keywords = [kw for kw in keywords if kw in content_lower]

        if matched_keywords:
            matches.append({
                "document": filename,
                "matched_keywords": matched_keywords,
                "relevance": len(matched_keywords) / len(keywords),
                "content": content
            })

    # Sort by relevance (most keyword matches first)
    matches.sort(key=lambda x: x["relevance"], reverse=True)

    if not matches:
        return json.dumps({"message": "No relevant documents found."})
    return json.dumps(matches, indent=2)


# Quick test: verify the functions work
print("=== Test search_orders ===")
print(search_orders(order_id=1003))
print()
print("=== Test search_products ===")
print(search_products(category="Electronics", in_stock_only=True))
print()
print("=== Test search_knowledge_base ===")
result = json.loads(search_knowledge_base("return policy"))
print(f"Found {len(result)} documents, top match: {result[0]['document']}")

## Define Tool Schemas

These schemas tell the model what tools are available, what parameters each tool accepts, and how to call them. We use `strict: True` to ensure the model always provides well-formed arguments.

In [ ]:
tools = [
    {
        "type": "function",
        "name": "search_orders",
        "description": (
            "Search for customer orders. You can filter by customer email, "
            "a specific order ID, or order status. Provide at least one parameter."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "customer_email": {
                    "type": ["string", "null"],
                    "description": "Customer email address to look up orders for."
                },
                "order_id": {
                    "type": ["integer", "null"],
                    "description": "Specific order ID to look up."
                },
                "status": {
                    "type": ["string", "null"],
                    "enum": ["pending", "shipped", "delivered", "refunded", None],
                    "description": "Filter orders by status."
                }
            },
            "required": ["customer_email", "order_id", "status"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "search_products",
        "description": (
            "Search for products in the catalog. You can filter by category, "
            "search by product name, or filter to only in-stock items."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "category": {
                    "type": ["string", "null"],
                    "description": "Product category to filter by (e.g. Electronics, Accessories, Furniture)."
                },
                "search_term": {
                    "type": ["string", "null"],
                    "description": "Keyword to search for in product names."
                },
                "in_stock_only": {
                    "type": "boolean",
                    "description": "If true, only return products that are currently in stock."
                }
            },
            "required": ["category", "search_term", "in_stock_only"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "search_knowledge_base",
        "description": (
            "Search the knowledge base of support documents including return policy, "
            "shipping information, FAQs, and pricing plans. Use this for policy "
            "questions or general information requests."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query - keywords describing what the customer is asking about."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        },
        "strict": True
    }
]

print(f"Defined {len(tools)} tools: {[t['name'] for t in tools]}")

## Tool Dispatch Function

This function routes tool calls from the model to the appropriate Python function. It also handles argument parsing and provides clear error messages.

In [ ]:
# Map tool names to their corresponding functions
TOOL_FUNCTIONS = {
    "search_orders": search_orders,
    "search_products": search_products,
    "search_knowledge_base": search_knowledge_base,
}


def dispatch_tool_call(name, args):
    """
    Route a tool call to the appropriate function.
    Returns the result as a string.
    """
    func = TOOL_FUNCTIONS.get(name)
    if func is None:
        return json.dumps({"error": f"Unknown tool: {name}"})

    try:
        # Filter out None values so they use function defaults
        filtered_args = {k: v for k, v in args.items() if v is not None}
        return func(**filtered_args)
    except Exception as e:
        return json.dumps({"error": str(e)})


# Quick test of dispatch
result = dispatch_tool_call("search_orders", {"order_id": 1001, "customer_email": None, "status": None})
print("Dispatch test (order 1001):")
print(result)

## Build the Agent Loop

The agent loop is the core pattern for building tool-using agents with the Responses API:

1. Send the user's question to the model along with the available tools.
2. If the model returns a **text response**, we are done -- that is the final answer.
3. If the model returns one or more **tool calls**, execute each tool and feed the results back.
4. Repeat until the model produces a final text response or we hit the turn limit.

This loop allows the model to chain multiple tool calls together. For example, it might first search for a customer's orders, then look up the return policy, and finally synthesize both pieces of information into a single answer.

In [ ]:
def run_agent(user_query, max_turns=10, verbose=True):
    """
    Run the customer support agent with the given query.

    Args:
        user_query: The customer's question.
        max_turns: Maximum number of model calls (safety limit).
        verbose: If True, print intermediate tool calls for debugging.

    Returns:
        The agent's final text response.
    """
    messages = [{"role": "user", "content": user_query}]

    instructions = (
        "You are a helpful customer support agent for an online store. "
        "Use the available tools to look up information before answering. "
        "Always cite which source you used (order database, product catalog, "
        "or knowledge base document). If you cannot find the answer, say so "
        "honestly and suggest how the customer can get help."
    )

    for turn in range(max_turns):
        response = client.responses.create(
            model=MODEL,
            instructions=instructions,
            input=messages,
            tools=tools,
        )

        # Separate tool calls from other output
        tool_calls = [item for item in response.output if item.type == "function_call"]

        # If no tool calls, the model has produced a final answer
        if not tool_calls:
            if verbose:
                print(f"  [Turn {turn + 1}] Final response (no tool calls)")
            return response.output_text

        # Process each tool call
        if verbose:
            print(f"  [Turn {turn + 1}] Model requested {len(tool_calls)} tool call(s):")

        for tool_call in tool_calls:
            args = json.loads(tool_call.arguments)
            if verbose:
                print(f"    -> {tool_call.name}({args})")

            result = dispatch_tool_call(tool_call.name, args)

            if verbose:
                # Show a truncated preview of the result
                preview = result[:200] + "..." if len(result) > 200 else result
                print(f"    <- {preview}")

            # Append the tool call and its result to the conversation
            messages.append(tool_call)
            messages.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": result,
            })

    return "Max turns reached. Please try a more specific question."


print("Agent loop is ready.")

## Test the Agent

Let's run a series of test queries that exercise each tool individually and then test multi-tool scenarios.

In [ ]:
# Test 1: Order lookup by ID (should use search_orders)
query = "What's the status of order #1003?"
print(f"{'='*60}")
print(f"QUERY: {query}")
print(f"{'='*60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")

In [ ]:
# Test 2: Product search by category (should use search_products)
query = "What products do you have in the Electronics category?"
print(f"{'='*60}")
print(f"QUERY: {query}")
print(f"{'='*60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")

In [ ]:
# Test 3: Knowledge base search (should use search_knowledge_base)
query = "What is your return policy?"
print(f"{'='*60}")
print(f"QUERY: {query}")
print(f"{'='*60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")

In [ ]:
# Test 4: Multi-tool query (should use search_orders AND search_knowledge_base)
query = (
    "I'm alice@example.com and I want to know about my orders "
    "and whether I can return any of them."
)
print(f"{'='*60}")
print(f"QUERY: {query}")
print(f"{'='*60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")

In [ ]:
# Test 5: Product search with constraints (should use search_products)
query = "Do you have any laptops in stock under $1000?"
print(f"{'='*60}")
print(f"QUERY: {query}")
print(f"{'='*60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")

## Summary

In this notebook we built a multi-source customer support agent that combines structured and unstructured data retrieval. Here are the key takeaways:

- **Multi-source retrieval**: Real support agents need access to different kinds of data. Our agent seamlessly queries a SQL database for transactional data and searches Markdown documents for policy information, choosing the right tool for each question.

- **Structured parameters over text-to-SQL**: Instead of letting the model generate raw SQL (which is fragile and risky), we defined tools with typed, constrained parameters. The model picks values like `category="Electronics"` or `order_id=1003`, and the Python functions build safe queries internally. This approach is more reliable, easier to test, and avoids SQL injection risks.

- **The agent loop pattern**: The core loop -- call the model, check for tool calls, execute tools, feed results back -- is the fundamental building block of agentic systems. It allows the model to chain multiple lookups before composing a final answer.

- **Combining data sources in one response**: The most powerful queries are those that require information from multiple sources (e.g., looking up a customer's orders AND checking the return policy). The agent loop naturally handles this by allowing the model to make sequential or parallel tool calls across different data sources.